In [1]:
# !pip install chromadb sentence-transformers


In [2]:
article = """
Trump Said His Plans Wouldn’t Touch the White House. Then the East Wing Came Down.
President Trump initially said the ballroom construction would not dismantle parts of the White House. His officials now say it was cheaper and more structurally sound to simply demolish the East Wing.

As roaring machinery tore down one side of the White House, President Trump acknowledged on Wednesday that he was having the entire East Wing demolished to make way for his 90,000-square-foot ballroom, a striking expansion of a project that is remaking the profile of one of the nation’s most iconic buildings.

Mr. Trump was unsentimental as news of the demolition spread. “It was never thought of as being much,” he said of the East Wing, which was home to the first lady’s office and spaces used for ceremonial purposes. “It was a very small building.”

The process of tearing down the East Wing was expected to be completed as soon as this weekend, two senior administration officials said, as Mr. Trump moved rapidly to carry out a passion project that he said was necessary to host state dinners and other events.

But the previously unannounced decision to demolish the East Wing was at odds with Mr. Trump’s previous statements about the project, and underscored his intention to blast through the sensibilities of many in Washington to continue putting a lasting imprint on the White House.

Advertisement


The president also said on Wednesday that the ballroom would cost $300 million, $100 million more than initially estimated.

“In order to do it properly, we had to take down the existing structure,” Mr. Trump said. He also said — somewhat cryptically — that “certain areas are being left.” But the two senior administration officials, who spoke on condition of anonymity to discuss the plans, confirmed that the entire East Wing was being demolished.

The West Wing and the White House residence, where the president lives, are not affected by the project, which is the largest renovation to the White House in decades.

When Mr. Trump first announced his plans for the ballroom, he pledged that the White House would not be touched by the construction.

“It won’t interfere with the current building. It’ll be near it but not touching it,” he said in July. “And pays total respect to the existing building, which I’m the biggest fan of.”


Upon further evaluation, the White House determined it was cheaper and more structurally sound to demolish the East Wing than to build an addition, one of the administration officials said.

On Wednesday, the Secret Service kept onlookers away as heavy machinery ripped away at hunks of the building.

The scope of the demolition, and Mr. Trump’s repeated promises that the White House itself would not be affected by the work, were in many ways symbolic of how he has conducted his presidency. On a variety of issues, Mr. Trump has blown past norms and traditions, often moving so quickly that it can be too late for courts, Congress or the public to catch up.

The planned size of the ballroom would transform the footprint of the White House campus. At 90,000 square feet, the ballroom would be nearly double the size of the White House residence, which is 55,000 square feet.

The ballroom is only the latest renovation plan that Mr. Trump has undertaken since he took office for the second time. He is also leaving his mark on the Oval Office, which now features many gilded flourishes. He also paved over the Rose Garden; erected huge flag poles on the White House grounds; and is planning to build an arch in front of Arlington National Cemetery in the style of the Arc de Triomphe.


Sara C. Bronin, a law professor at George Washington University who led the Advisory Council on Historic Preservation under former President Joseph R. Biden Jr., said that Mr. Trump’s decision to tear down the East Wing appeared to run afoul of the National Historic Preservation Act, which requires federal agencies to take into account the effects of their actions on historic places.

“The Trump administration’s shortsighted decision to start demolishing parts of the White House is exactly the kind of action the N.H.P.A. was passed to circumvent,” she said.

Mr. Trump has said that he is raising tens of millions of dollars in private donations to fund the project. The president plans to contribute some of his own money as well, though the amount has not been determined, one of the officials said.

“It’s being paid for 100 percent by me and some friends of mine,” Mr. Trump said.

The East Wing was built in 1902 during Theodore Roosevelt’s presidency as an extension to the White House, but was overhauled in the 1940s at the request of President Franklin D. Roosevelt.

It was built primarily to conceal an underground bunker, the Presidential Emergency Operations Center. It also added formal work space for the White House staff, including the offices of the first lady. Lorenzo Winslow was the architect of that addition.


It has also housed the White House Social Office and served as a headquarters for planning parties, state dinners and other events.

With the demolition of the East Wing goes a slice of history. It was where President Bill Clinton met secretly with Dick Morris, a political adviser, without his staff knowing. It was where Vice President Dick Cheney was hustled to a bunker after the Sept. 11 terrorist attacks. Mr. Trump was rushed there, too, during protests in 2020.

The White House on Tuesday did not answer questions about the bunker, but one of the administration officials said the new structure would also have enhanced security features.

Amid backlash to the demolition, the White House has defended its decision, publishing photos of past renovations and construction projects undertaken by presidents.

The White House says the ballroom, once finished, will have a seated capacity of 650 people, though Mr. Trump said recently it would hold 999 people.
"""




# Chunking

In [3]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

sentences = nltk.sent_tokenize(article)

chunks = []
current_chunk = ""
for sent in sentences:
    if len(current_chunk) + len(sent) < 500:
        current_chunk += " " + sent
    else:
        chunks.append(current_chunk.strip())
        current_chunk = sent
if current_chunk:
    chunks.append(current_chunk.strip())

print(f"Created {len(chunks)} chunks.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Created 15 chunks.


# Embedding

In [4]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
def get_embedding(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**tokens)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return emb

In [6]:
import numpy as np
embeddings = np.vstack([get_embedding(c) for c in chunks])
print(embeddings.shape)

(15, 768)


# Local Storage

In [7]:
chunk_texts = chunks
chunk_vectors = embeddings


## Local Retrival

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

query = "What is the argument?"
query_emb = get_embedding(query).reshape(1, -1)

scores = cosine_similarity(query_emb, chunk_vectors)[0]

top_k = 2
top_indices = np.argsort(scores)[::-1][:top_k]

In [23]:
chunk_texts[top_indices[0]]

'“And pays total respect to the existing building, which I’m the biggest fan of.”\n\n\nUpon further evaluation, the White House determined it was cheaper and more structurally sound to demolish the East Wing than to build an addition, one of the administration officials said. On Wednesday, the Secret Service kept onlookers away as heavy machinery ripped away at hunks of the building.'

In [24]:
chunk_texts[top_indices[1]]

'Trump Said His Plans Wouldn’t Touch the White House. Then the East Wing Came Down. President Trump initially said the ballroom construction would not dismantle parts of the White House. His officials now say it was cheaper and more structurally sound to simply demolish the East Wing.'

# Database Storage

In [9]:
# !pip install chromadb
import chromadb
client = chromadb.Client()
collection = client.create_collection(name="distilbert_embeddings")

# Add chunks
for i, c in enumerate(chunks):
    collection.add(ids=[f"id_{i}"], documents=[c], embeddings=[embeddings[i].tolist()])

## Database Retrival

In [13]:
query_emb = get_embedding("What is the argument?").tolist()
results = collection.query(query_embeddings=[query_emb], n_results=2)

In [15]:
results['documents']

[['“And pays total respect to the existing building, which I’m the biggest fan of.”\n\n\nUpon further evaluation, the White House determined it was cheaper and more structurally sound to demolish the East Wing than to build an addition, one of the administration officials said. On Wednesday, the Secret Service kept onlookers away as heavy machinery ripped away at hunks of the building.',
  '“In order to do it properly, we had to take down the existing structure,” Mr. Trump said. He also said — somewhat cryptically — that “certain areas are being left.” But the two senior administration officials, who spoke on condition of anonymity to discuss the plans, confirmed that the entire East Wing was being demolished. The West Wing and the White House residence, where the president lives, are not affected by the project, which is the largest renovation to the White House in decades.']]